In [1]:
#################################################
# 1. Load Credentials from .env
#################################################

import os
from dotenv import load_dotenv
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd

load_dotenv()

USER = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

print("Credentials loaded.")

Credentials loaded.


In [2]:
#################################################
# 2. DBRepo Client Setup
#################################################

from dbrepo.RestClient import RestClient

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

client = RestClient(
    BASE_URL,
    username=USER,
    password=PASSWORD
)

print("DBRepo client initialized.")

DBRepo client initialized.


Database found:
ID: 81d82941-cae0-4cba-b27d-1bd883dd713a
Name: data_stew_grp22_air_quality
Internal Name: data_stew_grp22_air_quality_gvui

Tables returned by Python client:


Using known table names confirmed in DBRepo UI.

All required tables exist.


In [7]:
#################################################
# 4. Read SQL View Definitions
#################################################

from pathlib import Path

VIEWS_SQL_PATH = Path("../docs/views.sql")

# Check file exists
if not VIEWS_SQL_PATH.exists():

    raise Exception(f"views.sql not found at: {VIEWS_SQL_PATH}")

# Read SQL file
with open(VIEWS_SQL_PATH, "r") as file:

    views_sql = file.read()

print("views.sql loaded successfully.\n")

print(views_sql[:1000])

views.sql loaded successfully.

-- =========================================================
-- views.sql
-- Air Quality Data Stewardship Project
-- TU Wien - WP2 T2.4 View Definitions
-- =========================================================


-- =========================================================
-- Remove existing views if they already exist
-- =========================================================

DROP VIEW IF EXISTS vw_air_quality_features;

DROP VIEW IF EXISTS vw_daily_pollution_summary;

DROP VIEW IF EXISTS vw_station_pollution_summary;



-- =========================================================
-- View: vw_air_quality_features
-- Description:
-- Denormalized ML-ready feature table combining
-- measurements, temporal information, and station metadata.
-- =========================================================

CREATE VIEW vw_air_quality_features AS

SELECT
    m.measurement_id,

    s.station_id,
    s.station_name,
    s.latitude,
    s.longitude,

    t.time

In [11]:
#################################################
# 5. Get or Create View: vw_air_quality_features
#################################################

VIEW_NAME = "vw_air_quality_features"

db_full = client.get_database(DB_ID)

print("Views available in database metadata:")

for view in db_full.views:
    print("-", view.name, view.id)

feature_view = None

for view in db_full.views:
    if view.name == VIEW_NAME:
        feature_view = view
        break

if feature_view is not None:
    print(f"\n{VIEW_NAME} already exists. Skipping creation.")
    print("View ID:", feature_view.id)

else:
    raise Exception(
        f"{VIEW_NAME} not found in DBRepo metadata. "
        "Do not recreate it while Python client cannot see tables. "
        "Check the Views tab in DBRepo UI or restart the kernel and client."
    )

Views available in database metadata:


Exception: vw_air_quality_features not found in DBRepo metadata. Do not recreate it while Python client cannot see tables. Check the Views tab in DBRepo UI or restart the kernel and client.

In [8]:
#################################################
# 5. Create View: vw_air_quality_features
#################################################

from dbrepo.api.dto import QueryDefinition

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

VIEW_NAME = "vw_air_quality_features"

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.measurement_id",
            "t_measurement.station_id",
            "t_measurement.time_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.wind_direction",
            "t_measurement.wind_speed",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure",
            "t_measurement.solar_radiation",
            "t_measurement.rain",

            "t_measurement.ben",
            "t_measurement.tol",
            "t_measurement.mxil"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

MalformedError: Failed to map subset: column(s) not found in database 0 != 20

In [ ]:
#################################################
# 6. Validate Feature View
#################################################

views = client.get_views(DB_ID)

feature_view = None

for view in views:

    if view.name == "vw_air_quality_features":

        feature_view = view

        break

if feature_view is None:

    raise Exception("View not found.")

print("View ID:")

print(feature_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    feature_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_view = client.get_view_data(

    DB_ID,

    feature_view.id,

    page=0,

    size=10
)

display(df_view)

Views found in database metadata:


Exception: vw_air_quality_features not found in database metadata.

In [8]:
views = client.get_views(DB_ID)

print("Number of views:", len(views))

for view in views:
    print(view.name, "|", view.id)

Number of views: 0


In [7]:
#################################################
# 7. Create View: vw_daily_pollution_summary
#################################################

from dbrepo.api.dto import QueryDefinition

VIEW_NAME = "vw_daily_pollution_summary"

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.time_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

MalformedError: Failed to map subset: column(s) not found in database 0 != 11

In [11]:
#################################################
# 8. Validate View: vw_daily_pollution_summary
#################################################

views = client.get_views(DB_ID)

daily_view = None

for view in views:

    if view.name == "vw_daily_pollution_summary":

        daily_view = view

        break

if daily_view is None:

    raise Exception("View not found.")

print("View ID:")

print(daily_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    daily_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_daily_view = client.get_view_data(

    DB_ID,

    daily_view.id,

    page=0,

    size=10
)

display(df_daily_view)

Exception: View not found.

In [10]:
#################################################
# 9. Create View: vw_station_pollution_summary
#################################################

from dbrepo.api.dto import QueryDefinition

VIEW_NAME = "vw_station_pollution_summary"

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.station_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure",

            "t_measurement.wind_speed",
            "t_measurement.wind_direction"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

MalformedError: Failed to map subset: column(s) not found in database 0 != 13

In [9]:
#################################################
# 10. Validate View: vw_station_pollution_summary
#################################################

views = client.get_views(DB_ID)

station_view = None

for view in views:

    if view.name == "vw_station_pollution_summary":

        station_view = view

        break

if station_view is None:

    raise Exception("View not found.")

print("View ID:")

print(station_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    station_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_station_view = client.get_view_data(

    DB_ID,

    station_view.id,

    page=0,

    size=10
)

display(df_station_view)

Exception: View not found.